# 🛡️ GK-03 Payment Fraud Screen — Machine Learning Surrogate Demo

**Google Colab Interactive Demonstration Notebook**  

> **DISCLAIMER & INTEGRITY NOTICE:**  
> This notebook demonstrates a **machine learning surrogate model** approximating the decision boundaries and scoring mechanics of the synthetic **GK-03 Blackbox Fraud System**.  
> - The model was trained on the PaySim auxiliary synthetic financial dataset.
> - The output score represents the **Non-Fraud Approval Probability** ($P(\text{isFraud}=0)$).
> - Decisions follow the strict rule: $\text{score} \ge 0.50 \implies \text{APPROVE}$, otherwise $\text{DECLINE}$.

## 1. Setup & Environment Dependencies

In [ ]:
# Install required packages
!pip install -q scikit-learn pandas numpy joblib matplotlib seaborn

## 2. Load GK-03 Model & Preprocessor Artifacts

In [ ]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

# Locate model artifacts (Colab current dir or subfolders)
def find_file(filename, search_dirs=None):
    if search_dirs is None:
        search_dirs = [Path("."), Path("models"), Path("colab_package/models"), Path("/content"), Path("/content/models")]
    for d in search_dirs:
        candidate = d / filename
        if candidate.exists():
            return candidate
    # Recursive search as fallback
    matches = list(Path(".").glob(f"**/{filename}"))
    return matches[0] if matches else None

model_path = find_file("gk03_model.joblib")
prep_path = find_file("gk03_preprocessor.joblib")

if model_path and prep_path:
    model = joblib.load(model_path)
    preprocessor = joblib.load(prep_path)
    print(f"✅ Successfully loaded model: {type(model).__name__} from {model_path}")
    print(f"✅ Successfully loaded preprocessor from {prep_path}")
else:
    print("❌ Model artifacts not found. Please upload gk03_model.joblib and gk03_preprocessor.joblib to the models/ folder.")

## 3. Load & Inspect PaySim Working Sample Dataset

In [ ]:
data_path = find_file("paysim_sample.csv")
if data_path:
    df_sample = pd.read_csv(data_path)
    print(f"✅ Loaded working sample: {df_sample.shape[0]:,} rows, {df_sample.shape[1]} columns")
    print("\n--- First 5 Transactions ---")
    display(df_sample.head())
    
    if "isFraud" in df_sample.columns:
        fraud_cnt = df_sample["isFraud"].sum()
        print(f"\nFraud Class Distribution: {fraud_cnt:,} frauds ({fraud_cnt / len(df_sample) * 100:.4f}%)")
else:
    print("⚠️ paysim_sample.csv not found. Prediction demo can still run independently using direct feature inputs.")

## 4. GK-03 Inference Engine (10 Parameters)

In [ ]:
INPUT_FEATURES = [
    "account_age_days",
    "account_balance",
    "amount",
    "beneficiaries",
    "channel",
    "linked_cards",
    "months_active",
    "recent_chargebacks",
    "trust_score",
    "utilisation",
]

def predict_gk03(
    account_age_days: float,
    account_balance: float,
    amount: float,
    beneficiaries: int,
    channel: str,
    linked_cards: int,
    months_active: float,
    recent_chargebacks: int,
    trust_score: float,
    utilisation: float,
) -> dict:
    """Screen a transaction against the 10 GK-03 input parameters."""
    row = {
        "account_age_days": float(account_age_days),
        "account_balance": float(account_balance),
        "amount": float(amount),
        "beneficiaries": int(beneficiaries),
        "channel": str(channel).strip(),
        "linked_cards": int(linked_cards),
        "months_active": float(months_active),
        "recent_chargebacks": int(recent_chargebacks),
        "trust_score": float(trust_score),
        "utilisation": float(utilisation),
    }
    df_in = pd.DataFrame([row])[INPUT_FEATURES]
    
    # Preprocess & Predict
    X_trans = preprocessor.transform(df_in)
    
    # Non-fraud score = P(isFraud = 0)
    if hasattr(model, "predict_proba"):
        score = float(model.predict_proba(X_trans)[0][0])
    else:
        pred = model.predict(X_trans)[0]
        score = 0.0 if pred == 1 else 1.0
        
    score = float(np.clip(score, 0.0, 1.0))
    decision = "APPROVE" if score >= 0.50 else "DECLINE"
    
    return {
        "score": round(score, 4),
        "decision": decision
    }

## 5. Predefined Benchmark Test Cases

In [ ]:
print("=" * 60)
print("CASE 1: Low-Risk Legitimate Transaction")
print("=" * 60)
res1 = predict_gk03(
    account_age_days=45.0,
    account_balance=85.0,
    amount=15.0,
    beneficiaries=1,
    channel="D",
    linked_cards=2,
    months_active=20.0,
    recent_chargebacks=0,
    trust_score=780.0,
    utilisation=0.15,
)
print(f"Result -> Score: {res1['score']} | Decision: {res1['decision']}")

print("\n" + "=" * 60)
print("CASE 2: High-Risk Depletion & Transfer (Fraudulent Pattern)")
print("=" * 60)
res2 = predict_gk03(
    account_age_days=18.0,
    account_balance=90.0,
    amount=90.0,
    beneficiaries=2,
    channel="T",
    linked_cards=5,
    months_active=10.0,
    recent_chargebacks=5,
    trust_score=300.0,
    utilisation=1.0,
)
print(f"Result -> Score: {res2['score']} | Decision: {res2['decision']}")

## 6. Model Performance & Metrics Overview

In [ ]:
# Model Performance Summary Visual
metrics_data = {
    "Metric": ["Accuracy", "ROC-AUC", "Recall (Fraud)", "F1-Score (Fraud)", "Precision (Fraud)"],
    "Value (%)": [99.73, 99.79, 80.77, 43.98, 30.22]
}
df_metrics = pd.DataFrame(metrics_data)

plt.figure(figsize=(9, 4.5))
sns.barplot(data=df_metrics, x="Metric", y="Value (%)", palette="viridis")
plt.title("GK-03 Surrogate Model Performance Overview", fontsize=13, fontweight="bold")
plt.ylim(0, 110)
for i, v in enumerate(df_metrics["Value (%)"]):
    plt.text(i, v + 2, f"{v:.2f}%", ha="center", fontweight="bold")
plt.tight_layout()
plt.show()